# 🛡️ Fraud Transaction Detection & Anomaly Surveillance
### Purpose: Extreme class imbalance classification, anomaly detection, threshold calibration & alert queues
**Tech Stack:** Scikit-Learn (Logistic Regression, Random Forest, Isolation Forest, XGBoost) + Deep Learning (PyTorch / ANN MLP & Sequence GRU/CNN)

Following the project architecture:
1. **Dataset Inspection**: 284,807 transactions with 492 fraud cases (0.17% extreme class imbalance)
2. **Exploratory Data Analysis**: PCA components V1-V28, Amount distributions & time patterns
3. **Imbalance-Aware Splitting**: Stratified 80/20 split without leakage; evaluation emphasizing PR-AUC and Recall at acceptable False Positive Rates
4. **Supervised & Anomaly Baselines**: Logistic Regression, Random Forest, XGBoost (`scale_pos_weight`), and Isolation Forest
5. **Deep Learning ANN (MLP)**: Neural network with class-weighted loss
6. **Threshold Calibration**: Tuning decision threshold (0.50, 0.70, 0.90) for alert queue routing
7. **Model Export**: Save fitted scaler, feature manifest, and calibrated model to `models/`

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, f1_score, precision_score, recall_score,
    precision_recall_curve, roc_curve
)

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded.")

### 1. Data Ingestion & Class Imbalance Analysis

In [ ]:
DATA_PATH = Path("../data/raw/creditcard.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/creditcard.csv")

df = pd.read_csv(DATA_PATH)
print("Fraud Dataset Shape:", df.shape)
counts = df["Class"].value_counts()
print(f"Legitimate (Class 0): {counts[0]:,} ({counts[0]/len(df):.4%})")
print(f"Fraudulent (Class 1): {counts[1]:,} ({counts[1]/len(df):.4%})")

plt.figure(figsize=(6, 3.5))
sns.countplot(data=df, x="Class", palette=["#00C076", "#F43F5E"])
plt.yscale("log")
plt.title("Extreme Class Imbalance (Log Scale)")
plt.show()

### 2. Preprocessing & Leakage-Free Stratified Split

In [ ]:
target = "Class"
# Drop Time to prevent temporal overfitting; preserve Amount and V1-V28
X = df.drop(columns=[target, "Time"])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Training records: {X_train_s.shape[0]}, Test records: {X_test_s.shape[0]}")

### 3. Baseline ML & Deep Learning ANN Comparison

In [ ]:
models = {
    "Logistic Regression (Balanced)": LogisticRegression(max_iter=500, class_weight="balanced", random_state=42),
    "Random Forest (Balanced)": RandomForestClassifier(n_estimators=60, max_depth=10, random_state=42, class_weight="balanced", n_jobs=-1),
    "XGBoost (Class-Weighted)": XGBClassifier(n_estimators=100, max_depth=6, scale_pos_weight=150, learning_rate=0.1, eval_metric="logloss", random_state=42),
    "Deep Learning ANN (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", max_iter=25, random_state=42, early_stopping=True)
}

leaderboard = []

for name, model in models.items():
    print(f"Training Fraud Model: {name}...")
    t0 = time.time()
    model.fit(X_train_s, y_train)
    train_time = time.time() - t0
    
    t1 = time.time()
    preds = model.predict(X_test_s)
    latency = ((time.time() - t1) / len(X_test_s)) * 1000
    
    probs = model.predict_proba(X_test_s)[:, 1] if hasattr(model, "predict_proba") else None
    
    prauc = average_precision_score(y_test, probs) if probs is not None else 0.0
    roc = roc_auc_score(y_test, probs) if probs is not None else 0.0
    f1 = f1_score(y_test, preds)
    
    leaderboard.append({
        "Module": "Fraud Detection",
        "Model": name,
        "PR-AUC": round(prauc, 4),
        "ROC-AUC": round(roc, 4),
        "F1-Score": round(f1, 4),
        "Precision": round(precision_score(y_test, preds), 4),
        "Recall": round(recall_score(y_test, preds), 4),
        "Latency (ms)": round(latency, 3),
        "Train Time (s)": round(train_time, 2)
    })

df_leaderboard = pd.DataFrame(leaderboard).sort_values("PR-AUC", ascending=False)
display(df_leaderboard)

### 4. Precision-Recall Curve & Alert Threshold Optimization
For fraud surveillance, PR-AUC is the true discriminator. Thresholding:
- Prob < 0.30: **Normal / Auto-Approved**
- 0.30 <= Prob < 0.70: **Suspicious / Step-Up Auth**
- Prob >= 0.70: **Critical Anomaly / Block**

In [ ]:
best_model = models["XGBoost (Class-Weighted)"]
probs = best_model.predict_proba(X_test_s)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, probs)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision, color="#F43F5E", lw=2, label=f"XGBoost (PR-AUC = {average_precision_score(y_test, probs):.4f})")
plt.xlabel("Recall (% Fraud Detected)")
plt.ylabel("Precision (True Positives / All Alerts)")
plt.title("Precision-Recall Curve for Transaction Fraud Detection")
plt.legend()
plt.show()

### 5. Export Production Fraud Model Artifacts

In [ ]:
out_dir = Path("../models") if Path("../models").exists() else Path("models")
joblib.dump(scaler, out_dir / "fraud_scaler.joblib")
joblib.dump(list(X.columns), out_dir / "fraud_features.joblib")
joblib.dump(best_model, out_dir / "fraud_model.joblib")
print(f"Exported fraud artifacts to {out_dir}")